In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# PATHS
# ============================================================

BASE = Path("../data/processed/primary")

CARRIER_SCORE = BASE / "carrier_anomaly_scores.csv"
OUTPATIENT_SCORE = BASE / "outpatient_anomaly_scores.csv"
INPATIENT_SCORE = BASE / "inpatient_anomaly_scores.csv"

CARRIER_DATA = BASE / "carrier_ml_ready.csv"
OUTPATIENT_DATA = BASE / "outpatient_ml_ready.csv"
INPATIENT_DATA = BASE / "inpatient_ml_ready.csv"

OUTPUT_FILE = BASE / "unified_anomaly_scores.csv"

# ============================================================
# HELPER
# ============================================================

def build_claim_score_file(
    score_file,
    data_file,
    claim_type,
    score_has_ids=False
):

    print("\n" + "=" * 80)
    print(f"PROCESSING {claim_type}")
    print("=" * 80)

    # --------------------------------------------------------
    # Read anomaly scores
    # --------------------------------------------------------

    scores = pd.read_csv(score_file)

    print("Score rows:", f"{len(scores):,}")
    print("Score columns:", list(scores.columns))

    if "ANOMALY_SCORE" not in scores.columns:
        raise ValueError(
            f"{claim_type}: ANOMALY_SCORE not found."
        )

    # --------------------------------------------------------
    # If IDs already exist in score file, use them.
    # Otherwise recover IDs from original ML-ready data.
    # --------------------------------------------------------

    if (
        "CLM_ID" in scores.columns
        and "DESYNPUF_ID" in scores.columns
    ):

        result = scores[
            [
                "CLM_ID",
                "DESYNPUF_ID",
                "ANOMALY_SCORE"
            ]
        ].copy()

        print("IDs found directly in score file.")

    else:

        print(
            "IDs not present in score file."
        )

        print(
            "Recovering IDs from corresponding ML-ready dataset..."
        )

        # ----------------------------------------------------
        # Only read the two identifier columns
        # ----------------------------------------------------

        ids = pd.read_csv(
            data_file,
            usecols=[
                "CLM_ID",
                "DESYNPUF_ID"
            ]
        )

        print(
            "ID rows:",
            f"{len(ids):,}"
        )

        # ----------------------------------------------------
        # Critical safety check
        # ----------------------------------------------------

        if len(ids) != len(scores):

            raise ValueError(
                f"{claim_type}: score rows ({len(scores):,}) "
                f"do not match ID rows ({len(ids):,})."
            )

        # ----------------------------------------------------
        # Attach IDs by original row position
        # ----------------------------------------------------

        result = ids.copy()

        result["ANOMALY_SCORE"] = (
            scores["ANOMALY_SCORE"].to_numpy()
        )

        del ids

    # --------------------------------------------------------
    # Claim type
    # --------------------------------------------------------

    result["CLAIM_TYPE"] = claim_type

    # --------------------------------------------------------
    # Normalize score within claim type
    # --------------------------------------------------------

    result["ANOMALY_PERCENTILE"] = (
        result["ANOMALY_SCORE"]
        .rank(method="average", pct=True)
        * 100
    )

    # --------------------------------------------------------
    # Claim-type rank
    # --------------------------------------------------------

    result["CLAIM_TYPE_RANK"] = (
        result["ANOMALY_SCORE"]
        .rank(
            method="min",
            ascending=False
        )
        .astype("int64")
    )

    # --------------------------------------------------------
    # Risk level
    # --------------------------------------------------------

    result["RISK_LEVEL"] = np.select(
        [
            result["ANOMALY_PERCENTILE"] >= 99.9,
            result["ANOMALY_PERCENTILE"] >= 99.0,
            result["ANOMALY_PERCENTILE"] >= 95.0
        ],
        [
            "CRITICAL",
            "HIGH",
            "MEDIUM"
        ],
        default="LOW"
    )

    print(
        "Completed:",
        f"{len(result):,}",
        "claims"
    )

    return result


# ============================================================
# PROCESS EACH DATASET
# ============================================================

carrier = build_claim_score_file(
    CARRIER_SCORE,
    CARRIER_DATA,
    "CARRIER"
)

outpatient = build_claim_score_file(
    OUTPATIENT_SCORE,
    OUTPATIENT_DATA,
    "OUTPATIENT"
)

inpatient = build_claim_score_file(
    INPATIENT_SCORE,
    INPATIENT_DATA,
    "INPATIENT"
)

# ============================================================
# COMBINE
# ============================================================

print("\n" + "=" * 80)
print("COMBINING CLAIM TYPES")
print("=" * 80)

unified = pd.concat(
    [
        carrier,
        outpatient,
        inpatient
    ],
    ignore_index=True
)

# Release individual dataframes
del carrier
del outpatient
del inpatient

# ============================================================
# GLOBAL RANK
# ============================================================

unified = unified.sort_values(
    "ANOMALY_PERCENTILE",
    ascending=False
).reset_index(drop=True)

unified["GLOBAL_ANOMALY_RANK"] = (
    np.arange(len(unified)) + 1
)

# Put useful columns first

first_cols = [
    "GLOBAL_ANOMALY_RANK",
    "CLAIM_TYPE",
    "CLAIM_TYPE_RANK",
    "CLM_ID",
    "DESYNPUF_ID",
    "ANOMALY_SCORE",
    "ANOMALY_PERCENTILE",
    "RISK_LEVEL"
]

remaining_cols = [
    c for c in unified.columns
    if c not in first_cols
]

unified = unified[
    first_cols + remaining_cols
]

# ============================================================
# VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("UNIFIED ANOMALY DATASET")
print("=" * 80)

expected_rows = (
    4_741_335
    + 790_790
    + 66_773
)

print(
    "Rows:",
    f"{len(unified):,}"
)

print(
    "Expected:",
    f"{expected_rows:,}"
)

print(
    "Row count correct:",
    len(unified) == expected_rows
)

print(
    "Missing scores:",
    unified["ANOMALY_SCORE"].isna().sum()
)

print(
    "Infinite scores:",
    np.isinf(
        unified["ANOMALY_SCORE"]
    ).sum()
)

print(
    "Missing claim IDs:",
    unified["CLM_ID"].isna().sum()
)

print("\nClaim type distribution:")

print(
    unified["CLAIM_TYPE"]
    .value_counts()
)

print("\nRisk distribution:")

print(
    unified["RISK_LEVEL"]
    .value_counts()
)

print("\nTop 20 unified anomalies:")

print(
    unified[
        [
            "GLOBAL_ANOMALY_RANK",
            "CLAIM_TYPE",
            "CLAIM_TYPE_RANK",
            "CLM_ID",
            "DESYNPUF_ID",
            "ANOMALY_SCORE",
            "ANOMALY_PERCENTILE",
            "RISK_LEVEL"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

# ============================================================
# SAVE
# ============================================================

unified.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 80)
print("UNIFIED ANOMALY SCORES SAVED")
print("=" * 80)

print(
    "Rows:",
    f"{len(unified):,}"
)

print(
    "File:",
    OUTPUT_FILE
)


PROCESSING CARRIER
Score rows: 4,741,335
Score columns: ['ANOMALY_SCORE']
IDs not present in score file.
Recovering IDs from corresponding ML-ready dataset...
ID rows: 4,741,335
Completed: 4,741,335 claims

PROCESSING OUTPATIENT
Score rows: 790,790
Score columns: ['ANOMALY_SCORE']
IDs not present in score file.
Recovering IDs from corresponding ML-ready dataset...
ID rows: 790,790
Completed: 790,790 claims

PROCESSING INPATIENT
Score rows: 66,773
Score columns: ['DESYNPUF_ID', 'CLM_ID', 'ANOMALY_SCORE']
IDs found directly in score file.
Completed: 66,773 claims

COMBINING CLAIM TYPES

UNIFIED ANOMALY DATASET
Rows: 5,598,898
Expected: 5,598,898
Row count correct: True
Missing scores: 0
Infinite scores: 0
Missing claim IDs: 0

Claim type distribution:
CLAIM_TYPE
CARRIER       4741335
OUTPATIENT     790790
INPATIENT       66773
Name: count, dtype: int64

Risk distribution:
RISK_LEVEL
LOW         5318952
MEDIUM       223956
HIGH          50375
CRITICAL       5615
Name: count, dtype: int64

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# PATHS
# ============================================================

BASE = Path("../data/processed/primary")

CARRIER_SCORE_FILE = BASE / "carrier_anomaly_scores.csv"
OUTPATIENT_SCORE_FILE = BASE / "outpatient_anomaly_scores.csv"
INPATIENT_SCORE_FILE = BASE / "inpatient_anomaly_scores.csv"

CARRIER_ID_FILE = BASE / "carrier_ml_ready.csv"
OUTPATIENT_ID_FILE = BASE / "outpatient_ml_ready.csv"
INPATIENT_ID_FILE = BASE / "inpatient_ml_ready.csv"

OUTPUT_FILE = BASE / "unified_anomaly_scores_v2.csv"


# ============================================================
# LOAD IDs WITHOUT DROPPING DUPLICATES
# ============================================================

def load_ids(file, claim_type):

    print("\n" + "=" * 80)
    print(f"LOADING IDS: {claim_type}")
    print("=" * 80)

    ids = pd.read_csv(
        file,
        usecols=["CLM_ID", "DESYNPUF_ID"],
        dtype={
            "CLM_ID": "string",
            "DESYNPUF_ID": "string"
        }
    )

    print("Rows:", f"{len(ids):,}")
    print(
        "Unique CLM_ID:",
        f"{ids['CLM_ID'].nunique():,}"
    )
    print(
        "Duplicate CLM_ID:",
        f"{ids['CLM_ID'].duplicated().sum():,}"
    )

    return ids


# ============================================================
# LOAD SCORE + MATCH TO ORIGINAL ROW ORDER
# ============================================================

def load_scores(score_file, id_file, claim_type):

    print("\n" + "=" * 80)
    print(f"PROCESSING {claim_type}")
    print("=" * 80)

    scores = pd.read_csv(
        score_file,
        usecols=["ANOMALY_SCORE"]
    )

    print(
        "Score rows:",
        f"{len(scores):,}"
    )

    ids = load_ids(
        id_file,
        claim_type
    )

    # --------------------------------------------------------
    # CRITICAL VALIDATION
    # --------------------------------------------------------
    #
    # The anomaly scores were generated row-by-row from the
    # ML-ready dataset.
    #
    # Therefore we attach scores using ORIGINAL ROW ORDER.
    #
    # We intentionally do NOT deduplicate CLM_ID.
    # --------------------------------------------------------

    if len(ids) != len(scores):

        raise ValueError(
            f"{claim_type}: score rows "
            f"({len(scores):,}) do not match "
            f"ML-ready rows ({len(ids):,})."
        )

    result = ids.copy()

    result["ANOMALY_SCORE"] = (
        scores["ANOMALY_SCORE"].to_numpy()
    )

    result["CLAIM_TYPE"] = claim_type

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    if result["ANOMALY_SCORE"].isna().any():
        raise ValueError(
            f"{claim_type}: missing anomaly scores."
        )

    if np.isinf(
        result["ANOMALY_SCORE"]
    ).any():
        raise ValueError(
            f"{claim_type}: infinite anomaly scores."
        )

    print(
        f"{claim_type} completed:",
        f"{len(result):,}",
        "claims"
    )

    return result


# ============================================================
# LOAD ALL THREE
# ============================================================

carrier = load_scores(
    CARRIER_SCORE_FILE,
    CARRIER_ID_FILE,
    "CARRIER"
)

outpatient = load_scores(
    OUTPATIENT_SCORE_FILE,
    OUTPATIENT_ID_FILE,
    "OUTPATIENT"
)

inpatient = load_scores(
    INPATIENT_SCORE_FILE,
    INPATIENT_ID_FILE,
    "INPATIENT"
)


# ============================================================
# NORMALIZE WITHIN CLAIM TYPE
# ============================================================

def normalize_scores(df):

    # Higher raw anomaly score = more anomalous.
    #
    # Rank within claim type so the three separate
    # Isolation Forest score scales become comparable.

    df["ANOMALY_PERCENTILE"] = (
        df["ANOMALY_SCORE"]
        .rank(
            method="average",
            pct=True
        ) * 100
    )

    # Unified anomaly score:
    # 0 = least anomalous
    # 100 = most anomalous within its claim type

    df["ANOMALY_RISK_SCORE"] = (
        df["ANOMALY_PERCENTILE"]
    )

    df["CLAIM_TYPE_RANK"] = (
        df["ANOMALY_SCORE"]
        .rank(
            method="first",
            ascending=False
        )
        .astype("int64")
    )

    return df


carrier = normalize_scores(carrier)
outpatient = normalize_scores(outpatient)
inpatient = normalize_scores(inpatient)


# ============================================================
# COMBINE
# ============================================================

print("\n" + "=" * 80)
print("COMBINING THREE ANOMALY MODELS")
print("=" * 80)

unified = pd.concat(
    [
        carrier,
        outpatient,
        inpatient
    ],
    ignore_index=True
)


# ============================================================
# GLOBAL ANOMALY RANK
# ============================================================

unified = unified.sort_values(
    [
        "ANOMALY_RISK_SCORE",
        "ANOMALY_SCORE"
    ],
    ascending=[
        False,
        False
    ]
).reset_index(drop=True)

unified["GLOBAL_ANOMALY_RANK"] = (
    np.arange(len(unified)) + 1
)


# ============================================================
# ANOMALY LEVEL
# ============================================================

unified["ANOMALY_LEVEL"] = pd.cut(
    unified["ANOMALY_RISK_SCORE"],
    bins=[
        -np.inf,
        50,
        75,
        90,
        100
    ],
    labels=[
        "LOW",
        "MEDIUM",
        "HIGH",
        "CRITICAL"
    ],
    include_lowest=True
)


# ============================================================
# FINAL COLUMN ORDER
# ============================================================

unified = unified[
    [
        "GLOBAL_ANOMALY_RANK",
        "CLAIM_TYPE",
        "CLAIM_TYPE_RANK",
        "CLM_ID",
        "DESYNPUF_ID",
        "ANOMALY_SCORE",
        "ANOMALY_PERCENTILE",
        "ANOMALY_RISK_SCORE",
        "ANOMALY_LEVEL"
    ]
]


# ============================================================
# FINAL VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("UNIFIED ANOMALY SCORE VALIDATION")
print("=" * 80)

expected_rows = (
    len(carrier)
    + len(outpatient)
    + len(inpatient)
)

print(
    "Rows:",
    f"{len(unified):,}"
)

print(
    "Expected:",
    f"{expected_rows:,}"
)

print(
    "Row count correct:",
    len(unified) == expected_rows
)

print(
    "Missing anomaly scores:",
    unified["ANOMALY_SCORE"].isna().sum()
)

print(
    "Infinite anomaly scores:",
    np.isinf(
        unified["ANOMALY_SCORE"]
    ).sum()
)

print(
    "Missing anomaly risk:",
    unified["ANOMALY_RISK_SCORE"].isna().sum()
)

print("\nClaim type distribution:")
print(
    unified["CLAIM_TYPE"]
    .value_counts()
)


# ============================================================
# SCORE DISTRIBUTION BY CLAIM TYPE
# ============================================================

print("\n" + "=" * 80)
print("ANOMALY RISK BY CLAIM TYPE")
print("=" * 80)

print(
    unified
    .groupby("CLAIM_TYPE")["ANOMALY_RISK_SCORE"]
    .describe(
        percentiles=[
            .50,
            .90,
            .95,
            .99,
            .999
        ]
    )
)


# ============================================================
# RISK DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("ANOMALY LEVEL DISTRIBUTION")
print("=" * 80)

print(
    unified["ANOMALY_LEVEL"]
    .value_counts()
    .sort_index()
)


# ============================================================
# TOP 25
# ============================================================

print("\n" + "=" * 80)
print("TOP 25 UNIFIED ANOMALIES")
print("=" * 80)

print(
    unified.head(25).to_string(
        index=False
    )
)


# ============================================================
# SAVE
# ============================================================

unified.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 80)
print("UNIFIED ANOMALY SCORE DATASET CREATED")
print("=" * 80)

print(
    "Rows:",
    f"{len(unified):,}"
)

print(
    "Columns:",
    len(unified)
)

print(
    "Saved:",
    OUTPUT_FILE
)


PROCESSING CARRIER
Score rows: 4,741,335

LOADING IDS: CARRIER
Rows: 4,741,335
Unique CLM_ID: 4,741,335
Duplicate CLM_ID: 0
CARRIER completed: 4,741,335 claims

PROCESSING OUTPATIENT
Score rows: 790,790

LOADING IDS: OUTPATIENT
Rows: 790,790
Unique CLM_ID: 779,815
Duplicate CLM_ID: 10,975
OUTPATIENT completed: 790,790 claims

PROCESSING INPATIENT
Score rows: 66,773

LOADING IDS: INPATIENT
Rows: 66,773
Unique CLM_ID: 66,705
Duplicate CLM_ID: 68
INPATIENT completed: 66,773 claims

COMBINING THREE ANOMALY MODELS

UNIFIED ANOMALY SCORE VALIDATION
Rows: 5,598,898
Expected: 5,598,898
Row count correct: True
Missing anomaly scores: 0
Infinite anomaly scores: 0
Missing anomaly risk: 0

Claim type distribution:
CLAIM_TYPE
CARRIER       4741335
OUTPATIENT     790790
INPATIENT       66773
Name: count, dtype: int64

ANOMALY RISK BY CLAIM TYPE
                count       mean        std       min        50%        90%  \
CLAIM_TYPE                                                                   

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# PATH
# ============================================================

BASE = Path("../data/processed/primary")

FILE = BASE / "unified_anomaly_scores_v2.csv"

EXPECTED_ROWS = 5_598_898

# ============================================================
# EXPECTED COLUMNS
# ============================================================

EXPECTED_COLUMNS = [
    "GLOBAL_ANOMALY_RANK",
    "CLAIM_TYPE",
    "CLAIM_TYPE_RANK",
    "CLM_ID",
    "DESYNPUF_ID",
    "ANOMALY_SCORE",
    "ANOMALY_PERCENTILE",
    "ANOMALY_RISK_SCORE",
    "ANOMALY_LEVEL"
]

# ============================================================
# HEADER VALIDATION
# ============================================================

print("=" * 90)
print("UNIFIED ANOMALY SCORE VALIDATION")
print("=" * 90)

header = pd.read_csv(FILE, nrows=0)

actual_columns = list(header.columns)

print("\nActual columns:")
print(actual_columns)

print("\nNumber of columns:", len(actual_columns))
print("Expected columns:", len(EXPECTED_COLUMNS))

missing_columns = [
    c for c in EXPECTED_COLUMNS
    if c not in actual_columns
]

unexpected_columns = [
    c for c in actual_columns
    if c not in EXPECTED_COLUMNS
]

print("\nMissing expected columns:", missing_columns)
print("Unexpected columns:", unexpected_columns)

if not missing_columns:
    print("Column structure: PASS")
else:
    print("Column structure: FAIL")


# ============================================================
# CHUNKED VALIDATION
# ============================================================

CHUNK_SIZE = 250_000

total_rows = 0

missing_values = {}
infinite_values = {}

claim_type_counts = {}
duplicate_claim_ids = set()

min_score = np.inf
max_score = -np.inf

min_percentile = np.inf
max_percentile = -np.inf

min_risk = np.inf
max_risk = -np.inf

previous_global_rank = 0
rank_errors = 0

risk_level_counts = {}

claim_type_rank_errors = 0

# Keep only IDs seen more than once.
# This avoids storing every claim ID.
seen_claim_ids = set()
duplicate_ids = set()

# ============================================================
# PROCESS CHUNKS
# ============================================================

for chunk_number, chunk in enumerate(
    pd.read_csv(
        FILE,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    rows = len(chunk)
    total_rows += rows

    # --------------------------------------------------------
    # Missing values
    # --------------------------------------------------------

    for col in EXPECTED_COLUMNS:

        if col not in chunk.columns:
            continue

        count = int(chunk[col].isna().sum())

        if count > 0:
            missing_values[col] = (
                missing_values.get(col, 0) + count
            )

    # --------------------------------------------------------
    # Infinite numeric values
    # --------------------------------------------------------

    numeric_cols = [
        "GLOBAL_ANOMALY_RANK",
        "CLAIM_TYPE_RANK",
        "ANOMALY_SCORE",
        "ANOMALY_PERCENTILE",
        "ANOMALY_RISK_SCORE"
    ]

    for col in numeric_cols:

        if col not in chunk.columns:
            continue

        values = pd.to_numeric(
            chunk[col],
            errors="coerce"
        )

        inf_count = int(
            np.isinf(values).sum()
        )

        if inf_count > 0:
            infinite_values[col] = (
                infinite_values.get(col, 0) + inf_count
            )

    # --------------------------------------------------------
    # Claim type distribution
    # --------------------------------------------------------

    for claim_type, count in (
        chunk["CLAIM_TYPE"]
        .value_counts(dropna=False)
        .items()
    ):

        key = str(claim_type)

        claim_type_counts[key] = (
            claim_type_counts.get(key, 0) + int(count)
        )

    # --------------------------------------------------------
    # Risk level distribution
    # --------------------------------------------------------

    for level, count in (
        chunk["ANOMALY_LEVEL"]
        .value_counts(dropna=False)
        .items()
    ):

        key = str(level)

        risk_level_counts[key] = (
            risk_level_counts.get(key, 0) + int(count)
        )

    # --------------------------------------------------------
    # Score ranges
    # --------------------------------------------------------

    scores = pd.to_numeric(
        chunk["ANOMALY_SCORE"],
        errors="coerce"
    )

    percentiles = pd.to_numeric(
        chunk["ANOMALY_PERCENTILE"],
        errors="coerce"
    )

    risk_scores = pd.to_numeric(
        chunk["ANOMALY_RISK_SCORE"],
        errors="coerce"
    )

    if scores.notna().any():
        min_score = min(
            min_score,
            scores.min()
        )

        max_score = max(
            max_score,
            scores.max()
        )

    if percentiles.notna().any():
        min_percentile = min(
            min_percentile,
            percentiles.min()
        )

        max_percentile = max(
            max_percentile,
            percentiles.max()
        )

    if risk_scores.notna().any():
        min_risk = min(
            min_risk,
            risk_scores.min()
        )

        max_risk = max(
            max_risk,
            risk_scores.max()
        )

    # --------------------------------------------------------
    # Validate global rank
    # --------------------------------------------------------

    ranks = pd.to_numeric(
        chunk["GLOBAL_ANOMALY_RANK"],
        errors="coerce"
    )

    expected_ranks = np.arange(
        total_rows - rows + 1,
        total_rows + 1
    )

    rank_errors += int(
        (~ranks.eq(expected_ranks)).sum()
    )

    # --------------------------------------------------------
    # Validate CLAIM_TYPE_RANK starts correctly
    # --------------------------------------------------------

    for claim_type, group in chunk.groupby(
        "CLAIM_TYPE",
        sort=False
    ):

        type_ranks = pd.to_numeric(
            group["CLAIM_TYPE_RANK"],
            errors="coerce"
        )

        # Ranks should be unique within claim type.
        if type_ranks.duplicated().any():
            claim_type_rank_errors += int(
                type_ranks.duplicated().sum()
            )

    # --------------------------------------------------------
    # Duplicate CLM_ID detection
    # --------------------------------------------------------

    for claim_id in chunk["CLM_ID"]:

        claim_id = str(claim_id)

        if claim_id in seen_claim_ids:
            duplicate_ids.add(claim_id)
        else:
            seen_claim_ids.add(claim_id)

    print(
        f"Validated {total_rows:,} rows..."
    )


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n" + "=" * 90)
print("VALIDATION RESULTS")
print("=" * 90)

# ------------------------------------------------------------
# Row count
# ------------------------------------------------------------

print("\nRows:", f"{total_rows:,}")
print("Expected:", f"{EXPECTED_ROWS:,}")
print("Row count correct:", total_rows == EXPECTED_ROWS)

# ------------------------------------------------------------
# Columns
# ------------------------------------------------------------

print("\nColumns:", len(actual_columns))
print("Expected:", len(EXPECTED_COLUMNS))

# ------------------------------------------------------------
# Missing values
# ------------------------------------------------------------

print("\nMissing values:")

if missing_values:
    for col, count in missing_values.items():
        print(
            f"  {col}: {count:,}"
        )
else:
    print("  None")

# ------------------------------------------------------------
# Infinite values
# ------------------------------------------------------------

print("\nInfinite values:")

if infinite_values:
    for col, count in infinite_values.items():
        print(
            f"  {col}: {count:,}"
        )
else:
    print("  None")

# ------------------------------------------------------------
# Duplicate CLM_ID
# ------------------------------------------------------------

print("\nUnique CLM_ID:", f"{len(seen_claim_ids):,}")
print("Duplicate CLM_ID:", f"{len(duplicate_ids):,}")

# ------------------------------------------------------------
# Claim type distribution
# ------------------------------------------------------------

print("\nClaim type distribution:")

for claim_type, count in sorted(
    claim_type_counts.items()
):

    print(
        f"  {claim_type:<12} {count:>12,}"
    )

# ------------------------------------------------------------
# Risk level distribution
# ------------------------------------------------------------

print("\nAnomaly level distribution:")

for level, count in sorted(
    risk_level_counts.items()
):

    print(
        f"  {level:<10} {count:>12,}"
    )

# ------------------------------------------------------------
# Score ranges
# ------------------------------------------------------------

print("\nAnomaly score range:")
print("  Minimum:", min_score)
print("  Maximum:", max_score)

print("\nAnomaly percentile range:")
print("  Minimum:", min_percentile)
print("  Maximum:", max_percentile)

print("\nAnomaly risk score range:")
print("  Minimum:", min_risk)
print("  Maximum:", max_risk)

# ------------------------------------------------------------
# Rank validation
# ------------------------------------------------------------

print("\nGlobal rank errors:", rank_errors)
print(
    "Global rank sequential:",
    rank_errors == 0
)

print(
    "Claim-type rank duplicate errors:",
    claim_type_rank_errors
)

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

all_good = (
    total_rows == EXPECTED_ROWS
    and len(actual_columns) == len(EXPECTED_COLUMNS)
    and not missing_columns
    and not missing_values
    and not infinite_values
    and len(duplicate_ids) == 0
    and rank_errors == 0
    and claim_type_rank_errors == 0
    and min_percentile >= 0
    and max_percentile <= 100
    and min_risk >= 0
    and max_risk <= 100
)

print("\n" + "=" * 90)

if all_good:
    print("UNIFIED ANOMALY DATASET VALIDATION: PASS")
else:
    print("UNIFIED ANOMALY DATASET VALIDATION: REVIEW REQUIRED")

print("=" * 90)

UNIFIED ANOMALY SCORE VALIDATION

Actual columns:
['GLOBAL_ANOMALY_RANK', 'CLAIM_TYPE', 'CLAIM_TYPE_RANK', 'CLM_ID', 'DESYNPUF_ID', 'ANOMALY_SCORE', 'ANOMALY_PERCENTILE', 'ANOMALY_RISK_SCORE', 'ANOMALY_LEVEL']

Number of columns: 9
Expected columns: 9

Missing expected columns: []
Unexpected columns: []
Column structure: PASS
Validated 250,000 rows...
Validated 500,000 rows...
Validated 750,000 rows...
Validated 1,000,000 rows...
Validated 1,250,000 rows...
Validated 1,500,000 rows...
Validated 1,750,000 rows...
Validated 2,000,000 rows...
Validated 2,250,000 rows...
Validated 2,500,000 rows...
Validated 2,750,000 rows...
Validated 3,000,000 rows...
Validated 3,250,000 rows...
Validated 3,500,000 rows...
Validated 3,750,000 rows...
Validated 4,000,000 rows...
Validated 4,250,000 rows...
Validated 4,500,000 rows...
Validated 4,750,000 rows...
Validated 5,000,000 rows...
Validated 5,250,000 rows...
Validated 5,500,000 rows...
Validated 5,598,898 rows...

VALIDATION RESULTS

Rows: 5,598,8

In [7]:
import pandas as pd
from pathlib import Path

BASE = Path("../data/processed/primary")
FILE = BASE / "unified_anomaly_scores_v2.csv"

CHUNK_SIZE = 250_000

total_rows = 0

seen_type_claim = set()
duplicate_type_claim = set()

seen_full_key = set()
duplicate_full_key = set()

claim_type_counts = {}

for chunk_number, chunk in enumerate(
    pd.read_csv(
        FILE,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    total_rows += len(chunk)

    # ========================================================
    # CLAIM TYPE + CLM_ID
    # ========================================================

    for claim_type, claim_id in zip(
        chunk["CLAIM_TYPE"],
        chunk["CLM_ID"]
    ):

        key = (
            str(claim_type),
            str(claim_id)
        )

        if key in seen_type_claim:
            duplicate_type_claim.add(key)
        else:
            seen_type_claim.add(key)

    # ========================================================
    # CLAIM TYPE + CLM_ID + BENEFICIARY
    # ========================================================

    for claim_type, claim_id, bene_id in zip(
        chunk["CLAIM_TYPE"],
        chunk["CLM_ID"],
        chunk["DESYNPUF_ID"]
    ):

        key = (
            str(claim_type),
            str(claim_id),
            str(bene_id)
        )

        if key in seen_full_key:
            duplicate_full_key.add(key)
        else:
            seen_full_key.add(key)

    # ========================================================
    # CLAIM TYPE COUNTS
    # ========================================================

    for claim_type, count in (
        chunk["CLAIM_TYPE"]
        .value_counts()
        .items()
    ):

        claim_type_counts[claim_type] = (
            claim_type_counts.get(claim_type, 0)
            + int(count)
        )

    print(
        f"Checked {total_rows:,} rows..."
    )


# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 90)
print("CLAIM IDENTITY VALIDATION")
print("=" * 90)

print("\nTotal rows:", f"{total_rows:,}")

print(
    "Unique CLAIM_TYPE + CLM_ID:",
    f"{len(seen_type_claim):,}"
)

print(
    "Duplicate CLAIM_TYPE + CLM_ID:",
    f"{len(duplicate_type_claim):,}"
)

print(
    "Unique CLAIM_TYPE + CLM_ID + DESYNPUF_ID:",
    f"{len(seen_full_key):,}"
)

print(
    "Duplicate full claim identity:",
    f"{len(duplicate_full_key):,}"
)

print("\nClaim type distribution:")

for claim_type, count in sorted(
    claim_type_counts.items()
):
    print(
        f"  {claim_type:<12} {count:>12,}"
    )

print("\n" + "=" * 90)

if len(duplicate_full_key) == 0:
    print("CLAIM IDENTITY VALIDATION: PASS")
else:
    print("CLAIM IDENTITY VALIDATION: REVIEW")

print("=" * 90)

Checked 250,000 rows...
Checked 500,000 rows...
Checked 750,000 rows...
Checked 1,000,000 rows...
Checked 1,250,000 rows...
Checked 1,500,000 rows...
Checked 1,750,000 rows...
Checked 2,000,000 rows...
Checked 2,250,000 rows...
Checked 2,500,000 rows...
Checked 2,750,000 rows...
Checked 3,000,000 rows...
Checked 3,250,000 rows...
Checked 3,500,000 rows...
Checked 3,750,000 rows...
Checked 4,000,000 rows...
Checked 4,250,000 rows...
Checked 4,500,000 rows...
Checked 4,750,000 rows...
Checked 5,000,000 rows...
Checked 5,250,000 rows...
Checked 5,500,000 rows...
Checked 5,598,898 rows...

CLAIM IDENTITY VALIDATION

Total rows: 5,598,898
Unique CLAIM_TYPE + CLM_ID: 5,587,855
Duplicate CLAIM_TYPE + CLM_ID: 11,043
Unique CLAIM_TYPE + CLM_ID + DESYNPUF_ID: 5,587,855
Duplicate full claim identity: 11,043

Claim type distribution:
  CARRIER         4,741,335
  INPATIENT          66,773
  OUTPATIENT        790,790

CLAIM IDENTITY VALIDATION: REVIEW


In [8]:
import pandas as pd
from pathlib import Path

BASE = Path("../data/processed/primary")
FILE = BASE / "unified_anomaly_scores_v2.csv"

CHUNK_SIZE = 250_000

# We only need these 4 columns
cols = [
    "CLAIM_TYPE",
    "CLM_ID",
    "DESYNPUF_ID",
    "ANOMALY_SCORE"
]

# ------------------------------------------------------------
# First pass: collect scores for duplicated identities
# ------------------------------------------------------------

records = {}

for chunk_number, chunk in enumerate(
    pd.read_csv(
        FILE,
        usecols=cols,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    for row in chunk.itertuples(index=False):

        key = (
            row.CLAIM_TYPE,
            str(row.CLM_ID),
            str(row.DESYNPUF_ID)
        )

        if key in records:
            records[key].append(float(row.ANOMALY_SCORE))
        else:
            records[key] = [float(row.ANOMALY_SCORE)]

    print(f"Checked {chunk_number} chunks...")


# ------------------------------------------------------------
# Analyze duplicate identities
# ------------------------------------------------------------

duplicate_groups = {
    key: scores
    for key, scores in records.items()
    if len(scores) > 1
}

different_score_groups = {}

for key, scores in duplicate_groups.items():

    # More than one distinct score
    if len(set(scores)) > 1:
        different_score_groups[key] = scores


# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("DUPLICATE CLAIM SCORE VALIDATION")
print("=" * 90)

print(
    "Duplicate claim identity groups:",
    f"{len(duplicate_groups):,}"
)

print(
    "Duplicate groups with different anomaly scores:",
    f"{len(different_score_groups):,}"
)

print(
    "Duplicate groups with identical anomaly scores:",
    f"{len(duplicate_groups) - len(different_score_groups):,}"
)

# ------------------------------------------------------------
# Show examples if different scores exist
# ------------------------------------------------------------

if different_score_groups:

    print("\nExamples with different scores:")

    for i, (key, scores) in enumerate(
        different_score_groups.items()
    ):

        print(
            key,
            "scores =",
            scores
        )

        if i >= 9:
            break

print("\n" + "=" * 90)

if len(different_score_groups) == 0:
    print("DUPLICATE SCORE VALIDATION: PASS")
    print(
        "All repeated claim identities have identical anomaly scores."
    )
else:
    print("DUPLICATE SCORE VALIDATION: REVIEW REQUIRED")

print("=" * 90)

Checked 1 chunks...
Checked 2 chunks...
Checked 3 chunks...
Checked 4 chunks...
Checked 5 chunks...
Checked 6 chunks...
Checked 7 chunks...
Checked 8 chunks...
Checked 9 chunks...
Checked 10 chunks...
Checked 11 chunks...
Checked 12 chunks...
Checked 13 chunks...
Checked 14 chunks...
Checked 15 chunks...
Checked 16 chunks...
Checked 17 chunks...
Checked 18 chunks...
Checked 19 chunks...
Checked 20 chunks...
Checked 21 chunks...
Checked 22 chunks...
Checked 23 chunks...

DUPLICATE CLAIM SCORE VALIDATION
Duplicate claim identity groups: 11,043
Duplicate groups with different anomaly scores: 11,043
Duplicate groups with identical anomaly scores: 0

Examples with different scores:
('OUTPATIENT', '542852281547374', '7C3AA9C0A4A8CB96') scores = [0.2003651892192975, 0.099914880398128]
('OUTPATIENT', '542902281474162', '7C3AA9C0A4A8CB96') scores = [0.189688072018346, 0.0660491990673219]
('OUTPATIENT', '542632281494557', '000B4662348C35B4') scores = [0.1820609666804891, 0.0647820235423157]
('OU

In [9]:
import pandas as pd
from pathlib import Path

BASE = Path("../data/processed/primary")

FILE = BASE / "outpatient_ml_ready.csv"

TARGET_CLM_ID = "542852281547374"
TARGET_BENE = "7C3AA9C0A4A8CB96"

# ============================================================
# READ ONLY THE COLUMNS WE NEED
# ============================================================

df = pd.read_csv(
    FILE,
    low_memory=False
)

# ============================================================
# FIND THE DUPLICATE CLAIM
# ============================================================

matches = df[
    (df["CLM_ID"].astype(str) == TARGET_CLM_ID) &
    (df["DESYNPUF_ID"].astype(str) == TARGET_BENE)
].copy()

print("=" * 90)
print("DUPLICATE CLAIM INVESTIGATION")
print("=" * 90)

print("\nRows found:", len(matches))

print("\nColumns:")
print(matches.columns.tolist())

print("\nMatching records:")
print(matches.to_string(index=False))

DUPLICATE CLAIM INVESTIGATION

Rows found: 2

Columns:
['CLAIM_KEY', 'DESYNPUF_ID', 'CLM_ID', 'CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT', 'TOTAL_REIMBURSEMENT', 'CLAIM_DURATION_DAYS', 'CLAIM_YEAR', 'CLAIM_MONTH', 'DIAGNOSIS_COUNT', 'PROCEDURE_COUNT', 'HCPCS_COUNT', 'HAS_DIAGNOSIS', 'HAS_PROCEDURE', 'HAS_HCPCS', 'HAS_NEGATIVE_PAYMENT', 'HAS_PRIMARY_PAYER_PAYMENT', 'IS_SEGMENT_2', 'HAS_SEGMENT_1_MATCH', 'BENE_SEX_IDENT_CD', 'BENE_RACE_CD', 'BENE_ESRD_IND', 'SP_STATE_CODE', 'BENE_COUNTY_CD', 'BENE_HI_CVRAGE_TOT_MONS', 'BENE_SMI_CVRAGE_TOT_MONS', 'BENE_HMO_CVRAGE_TOT_MONS', 'PLAN_CVRG_MOS_NUM', 'SP_ALZHDMTA', 'SP_CHF', 'SP_CHRNKIDN', 'SP_CNCR', 'SP_COPD', 'SP_DEPRESSN', 'SP_DIABETES', 'SP_ISCHMCHT', 'SP_OSTEOPRS', 'SP_RA_OA', 'SP_STRKETIA', 'MEDREIMB_IP', 'BENRES_IP', 'PPPYMT_IP', 'MEDREIMB_OP', 'BENRES_OP', 'PPPYMT_OP', 'MEDREIMB_CAR', 'BENRES_CAR', 'PPPYMT_CAR', 'CLM_PMT_AMT_MISSING', 'TOTAL_REIMBURSEMENT_MISSING', 'CLAIM_DURATION_DAYS_MISSING', 'DIAGNOSIS_COUNT_MISSING', 'PROCEDURE_COUNT

In [10]:
print("=" * 80)
print("CHECKING TRAINED ANOMALY MODELS")
print("=" * 80)

for name in [
    "carrier_model",
    "outpatient_model",
    "inpatient_model",
    "carrier_if",
    "outpatient_if",
    "inpatient_if",
    "carrier_scaler",
    "outpatient_scaler",
    "inpatient_scaler",
    "scaler",
]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {type(obj).__name__}")

print("=" * 80)

CHECKING TRAINED ANOMALY MODELS
